# ZeeChecker: train our own model (rough notes to fields)

1. **Runtime → Change runtime type → T4 GPU**
2. Add a Hugging Face **write** token as a Colab secret named `HF_TOKEN` (key icon on the left). Never paste it into a cell.
3. Run the cells from top to bottom. Nothing has to be uploaded: the made-up notes are made here, and they are the same every time.

Only made-up notes are used here, **no real reports**.

In [ ]:
%pip install -q "trl==1.18.2" "peft==0.22.0" "transformers==5.24.0"
%pip install -q python-dotenv pyyaml httpx

If Colab asks to restart the session after the install, restart it and continue with the next cell.

In [ ]:
import torch

assert torch.cuda.is_available(), "Choose a T4 GPU in Runtime -> Change runtime type"
print(torch.cuda.get_device_name())

# The code of the app. After its PR is merged, write "develop" or "main" here
BRANCH = "feat/notes-model"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
REPO_NAME = "zeechecker-notes-qwen2.5-1.5b-GGUF"
GGUF_FILE = "zeechecker-notes-q8_0.gguf"
TEST_NOTES = 200  # fewer for a quick try

## The made-up notes

In [ ]:
# Deleted first, because git can't clone into an existing folder and would keep the old code
!rm -rf /content/app && git clone -q --depth 1 --branch {BRANCH} https://github.com/mike1208-dev/team-doc-checker /content/app
!cd /content/app && python -m training.make_notes

In [ ]:
import sys

sys.path.insert(0, "/content/app")
from pathlib import Path

from datasets import load_dataset

from training.score_notes import app_prompt, read
from training.make_notes import to_chat

NOTES = Path("/content/app/data/notes")
tests = read(NOTES / "test_examples.jsonl")[:TEST_NOTES]
data = load_dataset("json", data_files={name: str(NOTES / f"{name}.jsonl") for name in ("train", "val")})
print(data, len(tests))
print(data["train"][0]["prompt"][0]["content"])
print(data["train"][0]["completion"][0]["content"])

## Scoring

The answers are made on the GPU, and scored by the app's own code. `wrong fact and no warning` is the most important number.

In [ ]:
import json
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")


def answer_all(model, name, short, batch=8):
    model.eval()
    rows = []
    for i in range(0, len(tests), batch):
        examples = tests[i : i + batch]
        prompts = [to_chat(e)["prompt"] if short else [{"role": "user", "content": app_prompt(e)}] for e in examples]
        inputs = tokenizer.apply_chat_template(prompts, add_generation_prompt=True, return_tensors="pt", return_dict=True, padding=True).to(model.device)
        started = time.perf_counter()
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=700, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        seconds = round((time.perf_counter() - started) / len(examples), 1)
        texts = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        rows += [{"id": e["id"], "text": text.strip().removeprefix("```json").removesuffix("```").strip(), "seconds": seconds} for e, text in zip(examples, texts)]
        print(f"{len(rows)} of {len(tests)}")
    (NOTES / f"answers_{name}.jsonl").write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")


def show_scores(name):
    !cd /content/app && python -m training.score_notes --name {name} --limit {TEST_NOTES}

## Before training

The untrained model gets the app's long prompt with all the rules, so the comparison is fair.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="cuda")
answer_all(model, "untrained", short=False, batch=4)
del model
torch.cuda.empty_cache()
show_scores("untrained")

## Train

Only the answer counts for the loss, not the prompt. The version with the lowest validation loss is kept.
If the GPU runs out of memory, use 1 for `per_device_train_batch_size` and 8 for `gradient_accumulation_steps`.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=BASE_MODEL,
    args=SFTConfig(
        output_dir="checkpoints",
        model_init_kwargs={"dtype": "float32"},  # with fp16 below; loading in fp16 breaks training on a T4
        fp16=True,
        bf16=False,
        num_train_epochs=3,
        learning_rate=2e-4,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        max_length=2048,
        completion_only_loss=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=20,
        report_to="none",
        seed=42,
    ),
    train_dataset=data["train"],
    eval_dataset=data["val"],
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
        task_type="CAUSAL_LM",
    ),
)
trainer.train()

## After training

In [ ]:
merged = trainer.model.merge_and_unload().to(torch.float16)
merged.config.use_cache = True
answer_all(merged, "trained", short=True)
show_scores("trained")

In [ ]:
import pandas as pd

scores = {name: read(NOTES / f"scores_{name}.json")[0] for name in ("untrained", "trained")}
display(pd.DataFrame(scores))

from google.colab import files

!cd /content/app/data/notes && zip -q /content/notes_results.zip answers_*.jsonl scores_*.json
files.download("/content/notes_results.zip")

## Make the model file (GGUF) and save it to a private Hugging Face repository

Saving is only worth it if the trained model is clearly better.

In [ ]:
merged.save_pretrained("merged")
tokenizer.save_pretrained("merged")

!rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q gguf sentencepiece
!python llama.cpp/convert_hf_to_gguf.py merged --outfile {GGUF_FILE} --outtype q8_0
!ls -lh {GGUF_FILE}

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
repo_id = f"{api.whoami()['name']}/{REPO_NAME}"
api.create_repo(repo_id, private=True, exist_ok=True)
assert api.model_info(repo_id).private, "The repository is public. Make it private first."
api.upload_file(path_or_fileobj=GGUF_FILE, path_in_repo=GGUF_FILE, repo_id=repo_id)
print(repo_id, GGUF_FILE)